# Montreal City Walkthrough

This notebook builds a Montreal-focused demo from the local `urban-energy-data` sibling plus the managed Montreal 3D building dataset on `Z:`.

It shows:
- what data pieces are available
- how to assemble a Montreal city object
- a small-sample workflow for visualization and analytics

The analytics demo uses roughly 5% of FSAs and 5% of buildings so it stays usable in a notebook.

In [ ]:
from pathlib import Path
import subprocess
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd().resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from urban_energy_core.io import (
    combine_montreal_building_sources,
    load_all_da_census,
    load_all_fsa_census,
    load_city_da_geojsons,
    load_city_fsa_geojsons,
    load_city_weather_csvs,
    load_montreal_building_geometry,
    load_montreal_building_inventory,
    load_processed_electricity_wide,
)
from urban_energy_core.pipelines import build_cities_from_data
from urban_energy_core.plotting import plot_spatial_samples_with_basemap


In [ ]:
DATA_ROOT = REPO_ROOT.parent / "urban-energy-data"
Z_3D_ROOT = Path(r"Z:\Public\Montreal 3D data")

ELEC_PATH = DATA_ROOT / "data" / "processed" / "electricity" / "elec_rebuilt_new_weather.parquet"
FSA_CENSUS_ROOT = DATA_ROOT / "data" / "raw" / "census" / "FSA scale"
DA_CENSUS_ROOT = DATA_ROOT / "data" / "raw" / "census" / "DA scale"
GEOMETRY_ROOT = DATA_ROOT / "data" / "raw" / "geometry"
WEATHER_ROOT = DATA_ROOT / "data" / "raw" / "weather"
BUILDINGS_ROOT = DATA_ROOT / "data" / "raw" / "buildings" / "montreal"

LOD1_PATH = BUILDINGS_ROOT / "LoD1.parquet"
EXT_BUILDING_GEO_PATH = Z_3D_ROOT / "Mtl_Buildings_Dec2022_KKv1.geojson"

for path in [DATA_ROOT, ELEC_PATH, FSA_CENSUS_ROOT, DA_CENSUS_ROOT, GEOMETRY_ROOT, WEATHER_ROOT, LOD1_PATH, EXT_BUILDING_GEO_PATH]:
    print(f"{path} -> {path.exists()}")

In [ ]:
subprocess.run([sys.executable, str(REPO_ROOT / "scripts" / "inspect_data_root.py")], check=False)

## Load Montreal Inputs

In [ ]:
elec_df = load_processed_electricity_wide(ELEC_PATH)

fsa_census_df = load_all_fsa_census(root_dir=FSA_CENSUS_ROOT, drop_key_col=False, show_progress=False)
if "GEO UID" in fsa_census_df.columns:
    fsa_census_df = fsa_census_df.set_index("GEO UID")
fsa_census_df.index = fsa_census_df.index.astype(str)

da_census_df = load_all_da_census(root_dir=DA_CENSUS_ROOT, drop_key_col=False, show_progress=False)
if "DAUID" in da_census_df.columns:
    da_census_df = da_census_df.set_index("DAUID")
da_census_df.index = da_census_df.index.astype(str)

fsa_geo = load_city_fsa_geojsons(geometry_dir=GEOMETRY_ROOT, show_progress=False)["montreal"]
da_geo = load_city_da_geojsons(geometry_dir=GEOMETRY_ROOT, show_progress=False)["montreal"]

montreal_boundary = fsa_geo.unary_union
da_geo = da_geo.loc[da_geo.geometry.centroid.within(montreal_boundary)].copy()
da_codes_montreal = pd.Index(da_geo["DAUID"].astype(str).unique())
da_census_df = da_census_df.loc[da_census_df.index.intersection(da_codes_montreal)].copy()
weather_df = load_city_weather_csvs(weather_dir=WEATHER_ROOT, show_progress=False)["montreal"]

building_inventory_df = load_montreal_building_inventory(LOD1_PATH)
building_geo_gdf = load_montreal_building_geometry(EXT_BUILDING_GEO_PATH)
if building_geo_gdf.crs != fsa_geo.crs:
    building_geo_gdf = building_geo_gdf.to_crs(fsa_geo.crs)

building_df = combine_montreal_building_sources(
    inventory_df=building_inventory_df,
    primary_geometry_gdf=building_geo_gdf,
)

summary_df = pd.DataFrame(
    {
        "dataset": [
            "FSA electricity",
            "FSA census",
            "DA census",
            "Montreal FSA geometry",
            "Montreal DA geometry",
            "Montreal weather",
            "Montreal building inventory+geometry",
        ],
        "rows": [
            len(elec_df),
            len(fsa_census_df),
            len(da_census_df),
            len(fsa_geo),
            len(da_geo),
            len(weather_df),
            len(building_df),
        ],
        "cols": [
            elec_df.shape[1],
            fsa_census_df.shape[1],
            da_census_df.shape[1],
            len(fsa_geo.columns),
            len(da_geo.columns),
            weather_df.shape[1],
            building_df.shape[1],
        ],
    }
)
summary_df

## Sample About 5% for the Interactive Demo

In [ ]:
SAMPLE_FRACTION = 0.05
RANDOM_SEED = 42

mtl_fsa_codes = sorted([code for code in fsa_geo["FSA"].astype(str).unique() if code in elec_df.columns])
n_fsa_sample = max(5, round(len(mtl_fsa_codes) * SAMPLE_FRACTION))
sampled_fsa_codes = sorted(pd.Series(mtl_fsa_codes).sample(n=n_fsa_sample, random_state=RANDOM_SEED).tolist())

sampled_building_df = building_df.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_SEED).reset_index(drop=True)
sampled_da_geo = da_geo.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_SEED).reset_index(drop=True)

sampled_fsa_geo = fsa_geo[fsa_geo["FSA"].astype(str).isin(sampled_fsa_codes)].copy()
sampled_fsa_census_df = fsa_census_df.loc[fsa_census_df.index.intersection(sampled_fsa_codes)].copy()
sampled_elec_df = elec_df.loc[:, sampled_fsa_codes].copy()

sampled_building_gdf = gpd.GeoDataFrame(sampled_building_df.dropna(subset=["geometry"]).copy(), geometry="geometry", crs=fsa_geo.crs)

print(f"Sampled FSAs: {len(sampled_fsa_codes)} / {len(mtl_fsa_codes)}")
print(f"Sampled buildings: {len(sampled_building_gdf)} / {len(building_df)}")
print(f"Sampled DAs: {len(sampled_da_geo)} / {len(da_geo)}")

## Visualize the Main Data Pieces

In [ ]:
plot_spatial_samples_with_basemap(
    fsa_gdf=sampled_fsa_geo,
    da_gdf=sampled_da_geo,
    building_gdf=sampled_building_gdf,
    figsize=(20, 6),
    zoom=11,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

weather_df.assign(date_time_local=pd.to_datetime(weather_df["date_time_local"])) \
    .set_index("date_time_local")["temperature"] \
    .resample("D").mean() \
    .plot(ax=axes[0], color="#0f766e", linewidth=1)
axes[0].set_title("Montreal Daily Mean Temperature")
axes[0].set_xlabel("")

sampled_building_df[["year_built", "height_m", "volume"]].apply(pd.to_numeric, errors="coerce").hist(ax=axes[1], bins=20)
axes[1].set_title("Sampled Building Attribute Distributions")

plt.tight_layout()

## Build Montreal City Objects

- `montreal_full`: full FSA energy/census/weather context plus all Montreal DA geometry/census and sampled buildings
- `montreal_demo`: sampled FSA slice for faster analytics

In [ ]:
cities_full = build_cities_from_data(
    elec_df=elec_df,
    city_geojsons={"montreal": fsa_geo},
    city_weather={"montreal": weather_df},
    census_df=fsa_census_df,
    city_da_geojsons={"montreal": da_geo},
    da_census_df=da_census_df,
    city_building_gdfs={"montreal": sampled_building_df},
    assign_building_units=False,
    show_progress=False,
)
montreal_full = cities_full["montreal"]

montreal_full.assign_building_units(overwrite=True)

cities_demo = build_cities_from_data(
    elec_df=sampled_elec_df,
    city_geojsons={"montreal": sampled_fsa_geo},
    city_weather={"montreal": weather_df},
    census_df=sampled_fsa_census_df,
    show_progress=False,
)
montreal_demo = cities_demo["montreal"]

print(f"Full Montreal city: {len(montreal_full.fsas)} FSAs, {len(montreal_full.das)} DAs, {len(montreal_full.buildings)} sampled buildings")
print(f"Demo Montreal city: {len(montreal_demo.fsas)} FSAs")

In [ ]:
example_fsa = montreal_demo.get_fsa(montreal_demo.list_fsa_codes()[0])
example_building = montreal_full.get_building(montreal_full.list_building_codes()[0])

pd.DataFrame(
    {
        "entity": ["example_fsa", "example_building"],
        "code": [example_fsa.code, example_building.code],
        "fsa_code": [example_fsa.code, example_building.fsa_code],
        "building_type": [None, example_building.building_type],
        "year_built": [None, example_building.year_built],
        "height_m": [None, example_building.height_m],
    }
)

## Run Package Capabilities on the Sampled FSA Slice

In [ ]:
normalized = example_fsa.normalize_for_weather(montreal_demo.weather)
per_capita = example_fsa.per_capita_consumption()
prism_summary = example_fsa.apply_heating_prism(montreal_demo.weather)
short_term_summary = example_fsa.short_term_metrics(show_progress=False)

print(f"Normalized points: {len(normalized)}")
print(f"Per-capita mean: {float(per_capita.mean()):.6f}")
print(f"Heating slope: {prism_summary['heating_slope_per_hdd']:.4f}")
print(f"Short-term rows: {len(short_term_summary)}")

In [ ]:
prism_table = montreal_demo.compute_prism_table(show_progress=False)
short_term_table = montreal_demo.compute_short_term_table(show_progress=False)

display(prism_table.head())
display(short_term_table.head())

In [ ]:
mean_load = sampled_elec_df.mean().rename("mean_load").reset_index().rename(columns={"index": "FSA"})
plot_gdf = sampled_fsa_geo.merge(mean_load, on="FSA", how="left")

fig, ax = plt.subplots(figsize=(7, 6))
plot_gdf.plot(column="mean_load", cmap="YlOrRd", legend=True, edgecolor="black", linewidth=0.7, ax=ax)
ax.set_title("Sampled Montreal FSAs by Mean Electricity Load")
ax.set_axis_off()
plt.tight_layout()

## Notes

- FSA analytics are the main energy-analysis path right now.
- DA geometry and census are available in the local data root, but DA electricity is not yet part of this notebook workflow.
- Buildings are included here as a sampled context layer and can already carry package methods if building-level energy is added later.